In [1]:
import geopandas as gpd
import pandas as pd
import numpy as np

In [2]:
crashes = pd.read_excel('../data/5 Year Clark County .xlsx', sheet_name='Crash Data')
crashes = crashes[crashes["CrashSeverity"].isin(["K", "A"])]

In [3]:
# Converting some YES/NO fields to Boolean
crashes["HitandRun"] = crashes["HitandRun"].map({'YES': True, 'NO': False})
crashes["AlcoholInvolved"] = crashes["AlcoholInvolved"].map({'YES': True, 'NO': False})
crashes["PedestrianInvolved"] = crashes["PedestrianInvolved"].map({'YES': True, 'NO': False})

In [4]:
# Cleaning up crash type labels which sometimes are all caps
crashes["VehCrashType"] = crashes["VehCrashType"].str.capitalize()

In [5]:
# Load the CSV into a pandas DataFrame
crashes = crashes[["CrashNum", "CrashDate", "CrashTime", "CrashType", "CrashSeverity", "POINT_Y", "POINT_X",
                   "AlcoholInvolved", "PedestrianInvolved", "NumInjured", "NumFatalities", "HitandRun", "VehCrashType"]]

# Create the GeoDataFrame
# Note: Use (longitude, latitude) order for points_from_xy
gdf = gpd.GeoDataFrame(
    crashes,
    geometry=gpd.points_from_xy(crashes.POINT_X, crashes.POINT_Y),
    crs="EPSG:26911"
)

In [6]:
# Loading in vehicles data
vehicles = pd.read_excel('../data/5 Year Clark County .xlsx', sheet_name='Vehicles Involved')

In [7]:
# Processes one row of crash data to return a list of factors attributes to all the vehicles in a crash

# Ignores these factors:
IGNORED_FACTORS = ["other", "unknown"]
# Replaces these factors with cleaned up text
CLEANED_FACTORS = {
    "failed to yield row": "Failed to yield",
    "oth improper driving": "Improper driving",
}
def get_vehicle_factors(row):
    v = vehicles[vehicles["CrashNum"] == row["CrashNum"]]
    factors = []
    # Multiple factors are separated by "|"
    for rec in v["Factors"].str.strip().str.split("|"):
        if type(rec) == list:
            for i in rec:
                if not pd.isna(i):
                    if len(i.strip()) > 0:
                        if i.strip().lower() not in IGNORED_FACTORS:
                            if i.strip().lower() in CLEANED_FACTORS:
                                factors.append(CLEANED_FACTORS[i.strip().lower()])
                            else:
                                factors.append(i.strip().capitalize())
        else:
            if not pd.isna(rec):
                if len(rec.strip()) > 0:
                    if rec.strip().lower() not in IGNORED_FACTORS:
                        if rec.strip().lower() in CLEANED_FACTORS:
                            factors.append(CLEANED_FACTORS[rec.strip().lower()])
                        else:
                            factors.append(rec.strip().capitalize())
    return factors

In [8]:
gdf["vehicle_factors"] = gdf.apply(get_vehicle_factors, axis=1)

In [9]:
# Filtering vehicle data to just motorcycles and similar vehicle types
vehicles = vehicles[vehicles["VehType"].isin(["MOTORCYCLE", "MOPED", "MINICYCLE", "MOTORBIKE", "MOTORSCOOTER"])]

In [10]:
# Returns true if a motorcycle was one of the vehicles involved in a crash
def get_motorcycle_involved(row):
    v = vehicles[vehicles["CrashNum"] == row["CrashNum"]]
    if len(v) > 0:
        return True
    return False

In [11]:
gdf["MotorcycleInvolved"] = gdf.apply(get_motorcycle_involved, axis=1)

In [12]:
# Reading the persons tab of the dataset and creates a view of the bicyclists
persons = pd.read_excel('../data/5 Year Clark County .xlsx', sheet_name='Persons Involved')
bicyclist = persons[persons["PersonType"].isin(["E-Bike", "Pedal Cyclist", "Other Cyclist"])]

In [13]:
# Returns a list of all person factors attributed to this crash.
# Maps some poorly formatted text to easier to read text
PERSON_FACTORS = {
    'driver inattention/distracted': "Distracted driving",
    'drug involvement': "Drug impairment",
    'obstructed view': "Obstructed view",
    "apparently fatigued/asleep": "Fatigued or asleep",
    'driver ill/injured': "Driver ill or injured",

}
def get_person_factors(row):
    p = persons[persons["CrashNum"] == row["CrashNum"]]
    full_string = p["PersonFactors"].to_string().lower()
    output = []
    for factor in PERSON_FACTORS.keys():
        if factor in full_string:
            output.append(PERSON_FACTORS[factor])
    return output

In [14]:
gdf["PersonFactors"] = gdf.apply(get_person_factors, axis=1)

In [15]:
# Combining person + vehicle factors into a single set for each entry (no duplicates)
gdf["vehicle_factors"] = gdf["vehicle_factors"].combine(gdf["PersonFactors"], lambda x, y: list(set(x + y)))

In [16]:
# Then, semicolon-separate the factors
gdf["vehicle_factors"] = gdf["vehicle_factors"].str.join('; ')

In [17]:
# The above code will create some empty string rows if there's no factors, so we replace those with None
gdf["vehicle_factors"] = gdf["vehicle_factors"].replace('', None)
gdf.drop("PersonFactors", inplace=True, axis=1)

In [18]:
# Returns true if a bicycle was one of the vehicles involved in a crash
def get_bicycle_involved(row):
    p = bicyclist[bicyclist["CrashNum"] == row["CrashNum"]]
    if len(p) > 0:
        return True
    return False

In [19]:
gdf["BicycleInvolved"] = gdf.apply(get_bicycle_involved, axis=1)

In [20]:
# Transform to WGS84
gdf = gdf.to_crs(epsg=4326)

In [21]:
# Re-mapping K's and A's to full string
crash_lookup = {"K": "Fatal", "A": "Serious Injury"}
gdf["severity"] = gdf["CrashSeverity"].map(crash_lookup)

In [22]:
gdf.drop(columns=["POINT_X", "POINT_Y", "CrashSeverity", "CrashType"], inplace=True)

In [23]:
gdf = gdf.replace(' ', np.nan)
gdf = gdf.dropna(subset=["CrashNum", "CrashDate", "CrashTime", "geometry"])

In [24]:
# Parse the date column
gdf['CrashDate'] = pd.to_datetime(gdf['CrashDate'])

# Convert CrashTime (e.g. 1701) to a time string (e.g. "17:01")
gdf['CrashTime_str'] = gdf['CrashTime'].astype(int).astype(str).str.zfill(4)  # pad to 4 digits
gdf['CrashTime_str'] = pd.to_datetime(gdf['CrashTime_str'], format='%H%M').dt.time

# Combine date + time into a single datetime column
gdf['date'] = pd.to_datetime(
    gdf['CrashDate'].astype(str) + ' ' + gdf['CrashTime_str'].astype(str)
)

In [25]:
gdf["lat"] = gdf["geometry"].y
gdf["lng"] = gdf["geometry"].x

In [26]:
gdf.drop(columns=["CrashDate", "CrashTime", "CrashTime_str", "geometry"], inplace=True)

In [27]:
gdf.to_csv("../data/clark_vz_data.csv", index=False)